# Milestone 2 lab, notebook 6 of 8: What you built has a name

## Where we are

| Notebook | What you do |
|---|---|
| 1. Ask only for what changed | Read the orders that changed, not all of them |
| 2. Remember where you stopped | Keep a watermark. See what happens when it moves too early |
| 3. When the API says no | Meet the rate limit and a server error. Wait, and try again |
| 4. Land it raw | Put every order in Postgres, exactly as it arrived |
| 5. Pull or push | A short lab: let the order system send the changes to you |
| **6. What you built has a name** (you are here) | Change data capture, and what this kind cannot see |
| 7. Counted twice | Run a few mornings. Watch the report go wrong |
| 8. The production version | The same logic as a tested program |

Your load asks: "which orders changed since my watermark?" Data engineers have a name for capturing changes. This short notebook gives you the name, and shows what your kind of it can and cannot see.

In [1]:
from lab_helpers import api, read_orders, set_morning, show_doc

## Step 1: The name

> "Change data capture (CDC) means you capture each change to a table as its own event. The change can be an insert, an update, or a delete."

There are three common ways to do it. Yours is the first.

In [2]:
show_doc("HLD.md", "What this load is called")

**From `docs/HLD.md`**

> ### What this load is called
> 
> Reading rows by "changed since my watermark" is **polling**: you ask the source again and again what is new. It is also the simplest kind of **change data capture (CDC)**: query-based CDC.
> 
> | Kind of CDC | How it works | Sees deletes? | Sees every update? |
> |---|---|---|---|
> | **Query-based** (this milestone) | Poll with an `updated_at` query | No | No. Only the latest state |
> | Trigger-based | The database writes each change to a change table | Yes | Yes |
> | Log-based | Read the database's own log of changes | Yes | Yes |
> 
> Milestone 3 meets a deleted order, and shows why the bottom row exists.

**Your `updated_at` watermark load is query-based CDC.** It is the simplest kind. It needs nothing from the source but a reliable `updated_at`.

The table says it does not see every update. Check that on Arden's own data.

## Step 2: What really happened, and what the query saw

The order system keeps a list of every change it ever made. It is a lab feature of this fake API, called the change feed. Use it only to **count** what happened between Monday's run and Tuesday's run.

In [3]:
def last_position():
    """How many changes the order system has made so far."""
    body = api.get("/changes", params={"after_pos": 0, "limit": 1}).json()
    while body["more"]:
        body = api.get("/changes", params={"after_pos": body["last_pos"], "limit": 500}).json()
    return body["last_pos"]

set_morning("2026-06-08")
monday_position = last_position()

set_morning("2026-06-09")
happened = api.get("/changes", params={"after_pos": monday_position, "limit": 500}).json()["data"]
polled, _ = read_orders(updated_since="2026-06-08T12:00:00Z")

print(f"Changes that happened between the two runs: {len(happened)}")
print(f"Rows the query-based load read:             {len(polled)}")

It is now Monday 2026-06-08 07:00:00 in Arden.
It is now Tuesday 2026-06-09 07:00:00 in Arden.
Changes that happened between the two runs: 88
Rows the query-based load read:             29


More than three times as many changes happened as the load read. Nothing is broken. The load reads the **latest state** of each changed order, not each change. Follow one order.

In [4]:
order_id = 1227
print(f"Order {order_id}: what happened")
for change in happened:
    if change["order_id"] == order_id:
        print(f"   {change['changed_at']}  {change['op']}  {change['after']['status']}")

print(f"\nOrder {order_id}: what the query-based load read")
for order in polled:
    if order["order_id"] == order_id:
        print(f"   {order['updated_at']}     {order['status']}")

Order 1227: what happened
   2026-06-08T12:24:54Z  c  placed
   2026-06-08T12:58:54Z  u  packed
   2026-06-08T13:50:54Z  u  out_for_delivery
   2026-06-08T14:32:54Z  u  delivered

Order 1227: what the query-based load read
   2026-06-08T14:32:54Z     delivered


The order was placed, packed, sent out and delivered. The load saw one row: delivered.

For Arden's morning report that is fine. The manager needs to know what yesterday's orders look like **now**. Nobody asked how long packing took.

## Step 3: When this kind is enough

| Your situation | A good choice |
|---|---|
| The table is small, and data one day old is fine | Reload the full table (milestone 1) |
| Rows are added or updated, `updated_at` is reliable, and nobody needs deletes | A watermark load: **query-based CDC** (this milestone) |
| Deletes matter, or you need every change, or you need minutes of delay | Log-based CDC |

Arden is in the middle row, almost. One thing in that row is not true for Arden: *nobody needs deletes*. A test order gets deleted from the order system on 13 June, and this load has no way to notice. You will meet it in milestone 3.

## A new word

| Word | Meaning | Here |
|---|---|---|
| **Change data capture (CDC)** | Capturing each change to a table: an insert, an update or a delete | The general idea |
| **Query-based CDC** | CDC by polling with an `updated_at` query. It sees the latest state, and it does not see deletes | The watermark load you built |

The definitions and the table of three kinds are from the D8Loop chapter [Change Data Capture (CDC)](https://www.d8loop.com/modules/knowledge-corner/data-pipeline/change-data-capture-cdc).

## What you now know

- Your watermark load has a name: query-based change data capture.
- It reads the latest state of each changed order. It does not see every step in between.
- It cannot see a row that was deleted.

**Say it in an interview:** "My pipeline does query-based CDC: it polls for rows where `updated_at` is at or after the watermark. It only sees the latest state, which is all a daily report needs, and it cannot see deletes. If deletes or every intermediate change mattered, I would move to log-based CDC."

Next: [notebook 7](07_counted_twice.ipynb) runs the pipeline for a few mornings. Something is wrong with the report.